### Analyze $c_p$ and $\tau_w$ distributions

In [ ]:
import matplotlib.pyplot as plt
import torch as pt

from os import makedirs
from os.path import join, exists

from utils import compute_camber_line
from flowtorch.data import CSVDataloader

In [ ]:
# flow quantities
# u_inf = 238.845

# validation @ Ma = 0.73
u_inf = 242.16629

# angle of attack
aoa = 2.5

# chord length
chord = 0.15

# use latex fonts
plt.rcParams.update({"text.usetex": True, "figure.dpi": 360, "text.latex.preamble": r"\usepackage{xcolor}"})

# use default color cycle
color = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f"]

# use these linestyles
ls = ["-", "--", "-.", ":"]

In [ ]:
# define load and save paths
load_dir = join("..", "run", "URANS")

# save_dir = join("..", "run", "plots", "URANS_validation", "grid_convergence_study", f"AoA{aoa}deg")
save_dir = join("..", "run", "plots", "URANS_validation", "grid_convergence_study", "comparison_AoA")

cases = [f"URANS_kOmegaSST_alpha{aoa}deg_coarseGrid", f"URANS_kOmegaSST_alpha{aoa}deg",
         f"URANS_kOmegaSST_alpha{aoa}deg_fineGrid", f"URANS_kOmegaSST_alpha{aoa}deg_veryFineGrid"]

cases = ["URANS_kOmegaSST_alpha2.5deg", "URANS_kOmegaSST_alpha3.5deg"]

legend = [r"$\alpha = 2.5^\circ$ (coarse grid)", r"$\alpha = 2.5^\circ$ (default grid)",
          r"$\alpha = 2.5^\circ$ (fine grid)", r"$\alpha = 2.5^\circ$ (very fine grid)"]

# legend = [r"$\alpha = 3.5^\circ$ (coarse grid)", r"$\alpha = 3.5^\circ$ (default grid)",
#           r"$\alpha = 3.5^\circ$ (fine grid)", r"$\alpha = 3.5^\circ$ (very fine grid)"]

legend = [r"$\alpha = 2.5^\circ$ (default grid)", r"$\alpha = 3.5^\circ$ (default grid)"]

# write_times = ["0.058", "0.063"]
field = "total(p)_coeff"
# field = "wallShearStress"
save_name = "cp"

# create plot directory
if not exists(save_dir):
    makedirs(save_dir)

In [ ]:
x, z, cp = [], [], []
for c in cases:
    # TODO: refactor so mean cp or other quantitiy can be computed as well as instantaneous quantities
    loader = CSVDataloader.from_foam_surface(join(load_dir, c, "postProcessing", "surface"),
                                                 f"{field}_airfoil.raw")

    x_temp = loader.vertices[:, 0]
    z_temp = loader.vertices[:, 2]

    # compute dummy camber line to separate suction- from pressure side based on original x-coordinates of the af
    x_camber, camber_line = compute_camber_line(x_temp)

    # get all coordinates for suction and pressure side
    is_suction = z_temp > camber_line
    is_pressure = ~is_suction

    _field = field if field != "wallShearStress" else "wallShearStress_x"

    # take all times starting at t* = 50 to compute the mean cp TODO: automate, plot both mean and min/max
    write_times = loader.write_times[30:]
    
    cp_temp = loader.load_snapshot(_field, write_times).mean(dim=1)
    cp_suction = cp_temp[is_suction]
    cp_pressure = cp_temp[is_pressure]

    x_suction = x_temp[is_suction]
    z_suction = z_temp[is_suction]
    x_pressure = x_temp[is_pressure]
    z_pressure = z_temp[is_pressure]

    idx_suction = pt.argsort(x_suction, descending=True)
    x_suction = x_suction[idx_suction]
    z_suction = z_suction[idx_suction]
    cp_suction = cp_suction[idx_suction]

    idx_pressure = pt.argsort(x_pressure)
    x_pressure = x_pressure[idx_pressure]
    z_pressure = z_pressure[idx_pressure]
    cp_pressure = cp_pressure[idx_pressure]

    """
    # sanity check
    fig, ax = plt.subplots(figsize=(6, 2))
    ax.plot(x_pressure / chord, z_pressure / chord, color=color[0])
    ax.plot(x_suction / chord, z_suction / chord, color=color[2])
    ax.plot(x_camber / chord, camber_line / chord, color=color[1])
    ax.set_aspect("equal")
    ax.set_xlabel("$x / c$")
    ax.set_ylabel("$z / c$")
    fig.tight_layout()
    plt.savefig(join(save_dir, "airfoil_ps_ss_split.png"))
    plt.show()
    plt.close("all")
    exit()
    """

    x_sorted = pt.cat([x_suction, x_pressure])
    z_sorted = pt.cat([z_suction, z_pressure])
    cp_sorted = pt.cat([cp_suction, cp_pressure])

    x.append(x_sorted)
    z.append(z_sorted)
    cp.append(cp_sorted)

In [ ]:
# test plot airfoil with computed camber line
fig, ax = plt.subplots(figsize=(6, 2))
ax.plot(x[-1] / chord, z[-1] / chord, color=color[0])
ax.scatter(x_camber / chord, camber_line / chord, color=color[1], marker=".", s=2)
ax.set_aspect("equal")
ax.set_xlabel("$x / c$")
ax.set_ylabel("$z / c$")
fig.tight_layout()
plt.savefig(join(save_dir, "airfoil.png"))

In [ ]:
# plot mean cp-distribution
fig, ax = plt.subplots(figsize=(6, 4))
for j in range(len(cases)):
    ax.plot(x[j]/chord, cp[j], zorder=10, color=color[j], label=f"{legend[j]}")
ax.set_xlabel(r"$x/c$")
ax.set_ylabel("$c_p$")
ax.grid(visible=True, which="major", linestyle="-", alpha=0.35, color="black", axis="both")
ax.minorticks_on()
ax.tick_params(axis="x", which="minor", bottom=False)
ax.grid(visible=True, which="minor", linestyle="--", alpha=0.25, color="black", axis="both")
plt.gca().invert_yaxis()
fig.tight_layout()
fig.legend(ncol=2, loc="upper center")
fig.subplots_adjust(top=0.88)
plt.savefig(join(save_dir, f"comparison_mean_{save_name}_distribution.png"))

In [ ]:
# plot cp-distribution at different points in time
fig, ax = plt.subplots(figsize=(6, 4))
for j in range(len(cases)):
    for i in range(cp[j].size(-1)):
        ax.plot(x[j]/chord, cp[j][:, i], ls=ls[i], zorder=10, color=color[j],
                label=rf"{legend[j]} $(t^* = " + "{:.0f}".format(float(write_times[i]) * u_inf / chord) + ")$")
ax.set_xlabel(r"$x/c$")
ax.set_ylabel("$c_p$")
ax.grid(visible=True, which="major", linestyle="-", alpha=0.35, color="black", axis="both")
ax.minorticks_on()
ax.tick_params(axis="x", which="minor", bottom=False)
ax.grid(visible=True, which="minor", linestyle="--", alpha=0.25, color="black", axis="both")
plt.gca().invert_yaxis()
fig.tight_layout()
fig.legend(ncol=2, loc="upper center")
fig.subplots_adjust(top=0.8)
plt.savefig(join(save_dir, f"comparison_{save_name}_distribution.png"))